# Система массового обслуживания M/M/1

**Цель:** выполнить дискретно-событийное моделирование одноканальной СМО,
сравнить статистические оценки с формулами теории очередей.

In [1]:
using DrWatson
@quickactivate "project"
ENV["GKSwstype"] = "100"
using CSV, DataFrames, Plots, Statistics
include(srcdir("queueing_models.jl"))
using .QueueingModels

name = "01_mm1_queue"
mkpath(datadir(name)); mkpath(plotsdir(name))

lambda, mu = 30.0, 33.0
result = simulate_mm1(lambda, mu; horizon=1000.0, capacity=10_000, seed=202603)
theory = theoretical_mm1(lambda, mu)

summary = DataFrame(metric=["rho", "L", "Lq", "W", "Wq"],
    theory=[theory.rho, theory.mean_system, theory.mean_queue, theory.mean_sojourn, theory.mean_wait],
    simulation=[result.utilization, result.mean_system, result.mean_queue, result.mean_sojourn, result.mean_wait])
summary.relative_error = abs.((summary.simulation .- summary.theory) ./ summary.theory)
CSV.write(datadir(name, "summary.csv"), summary)
CSV.write(datadir(name, "waits.csv"), DataFrame(wait=result.waits))

println("=== СМО M/M/1: базовый эксперимент ===")
println("lambda = $lambda, mu = $mu, rho = $(round(theory.rho; digits=4))")
println("Заявок: $(result.arrivals), обслужено: $(result.served), потеряно: $(result.lost)")
show(summary; allrows=true, allcols=true); println()

default(fontfamily="DejaVu Sans", linewidth=2.4, framestyle=:box, gridalpha=0.22,
    left_margin=5 * Plots.mm)
limit = searchsortedlast(result.times, 20.0)
p1 = plot(result.times[1:limit], result.queue_lengths[1:limit]; seriestype=:steppost,
    xlabel="Время", ylabel="Длина очереди", label="Q(t)",
    title="Фрагмент траектории очереди M/M/1", size=(1000,620))
savefig(p1, plotsdir(name, "queue_trajectory.png"))

p2 = histogram(result.waits; bins=50, normalize=:pdf, alpha=0.65,
    xlabel="Время ожидания", ylabel="Плотность", label="моделирование",
    title="Распределение времени ожидания", size=(1000,620))
savefig(p2, plotsdir(name, "wait_histogram.png"))

p3 = plot(summary.metric, summary.theory; marker=:circle, markersize=8,
    label="теория", ylabel="Значение", title="Теория и имитация M/M/1",
    size=(1000,620))
plot!(p3, summary.metric, summary.simulation; marker=:diamond, markersize=7,
    linestyle=:dash, label="моделирование")
savefig(p3, plotsdir(name, "theory_vs_simulation.png"))

=== СМО M/M/1: базовый эксперимент ===


lambda = 30.0, mu = 33.0, rho = 0.9091
Заявок: 29825, обслужено: 29825, потеряно: 0
5×4 DataFrame
 Row │ metric  theory     simulation  relative_error 
     │ String  Float64    Float64     Float64        
─────┼───────────────────────────────────────────────
   1 │ rho      0.909091    0.914581      0.00603923
   2 │ L       10.0         9.52472       0.047528
   3 │ Lq       9.09091     8.61014       0.0528848
   4 │ W        0.333333    0.319354      0.0419393
   5 │ Wq       0.30303     0.288689      0.0473275


"/workspace/labs/lab03/project/plots/01_mm1_queue/theory_vs_simulation.png"

Полученные оценки близки к стационарным формулам; небольшое отличие связано
с конечной длительностью одного случайного прогона.